### ==============================================================================
## Processing of Moving Vessel Profiler Data -  Plot (CTD vs raw MVPs)
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from FaSt-SWOT / BioSWOT experiment
# 
**DESCRIPTION**:
 This script generates a comprehensive 5-panel validation figure comparing 
 a reference CTD cast with the 3 spatially and temporally closest MVP profiles 
 (the best match, plus the immediate ±1 profiles in the sequence). 
 The panels display:
   (a) Temperature, (b) Conductivity, (c) Salinity, 
   (d) Spatial Context Map, and (e) Salinity Difference (Delta S).

### ============================

## FaSt-SWOT

LEG 1:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import re
import io
import warnings
import gsw

warnings.filterwarnings("ignore")

# ==========================================
# 1. PARAMETERS
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")
DIR_MVP_RAW = BASE_ROOT / "Data" / "Leg1" / "processed_step1_highres_qc" 
DIR_CTD = Path(r"C:/Users/ASUS/OneDrive - Universitat de les Illes Balears/SWOT/CTD/CTD_data/leg1/HM/")

MAX_DIST_KM = 0.5
MAX_TIME_MIN = 60.0

#  Visual Configuration
Z_MAX_PROFILE = 100.0
Z_RANGE_DIFF = (10, 100)
SAL_LIMITS = (38.0, 38.5)

OUTDIR = BASE_ROOT / "Figures" / "figure1_paper_bioswot"
OUTDIR.mkdir(exist_ok=True)

# ==========================================
# 2. FUNCTIONS
# ==========================================
def parse_cnv_time(line):
    try:
        parts = line.split('=')[1].strip().split('[')[0].strip()
        return pd.to_datetime(parts)
    except: return pd.NaT

def read_ctd_cnv(path):
    try:
        with open(path, 'r', encoding='latin-1') as f: lines = f.readlines()
        lat, lon, time_val = np.nan, np.nan, pd.NaT
        start_idx = 0
        col_map = {}
        name_re = re.compile(r"#\s*name\s*(\d+)\s*=\s*([^:]+)", re.I)
        
        for i, line in enumerate(lines):
            if 'NMEA Latitude' in line:
                parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(parts)>=2: lat = float(parts[0]) + float(parts[1])/60 * (-1 if 'S' in line else 1)
            if 'NMEA Longitude' in line:
                parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(parts)>=2: lon = float(parts[0]) + float(parts[1])/60 * (-1 if 'W' in line else 1)
            if 'start_time' in line: time_val = parse_cnv_time(line)
            m = name_re.search(line)
            if m:
                idx = int(m.group(1)); name = m.group(2).strip().lower()
                if any(x in name for x in ['prdm', 'pres']): col_map['p'] = idx
                elif any(x in name for x in ['t090c', 'temp']): col_map['t'] = idx
                elif any(x in name for x in ['sal00', 'psal']): col_map['s'] = idx
            if '*END*' in line: start_idx = i+1; break
            
        data_str = "".join(lines[start_idx:])
        df = pd.read_csv(io.StringIO(data_str), delim_whitespace=True, header=None, on_bad_lines='skip')
        out = {'lat': lat, 'lon': lon, 'time': time_val, 'file': path.name}
        for k, idx in col_map.items(): 
            out[k] = df.iloc[:, idx].values if idx < df.shape[1] else np.nan
            
        # Recalcular Conductividad (mS/cm)
        if 's' in out and 't' in out and 'p' in out:
            out['c'] = gsw.C_from_SP(out['s'], out['t'], out['p'])
            
        return out
    except: return None

def load_mvp_catalog_sorted(mvp_dir):
    files = sorted(list(mvp_dir.glob("*.nc")))
    data = []
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                lat = np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values)
                lon = np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values)
                t_val = pd.NaT
                if 'start_time' in ds: t_val = pd.to_datetime(ds.start_time)
                elif 'time' in ds: 
                    ts = ds.time.values
                    if ts.size > 0: t_val = pd.to_datetime(ts.ravel()[0])
                data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val, 'path': f})
        except: pass
    return pd.DataFrame(data)

# ==========================================
# 3. MATCHING
# ==========================================
def find_closest_sequence_strict(ctd, df_mvp):
    if df_mvp.empty or pd.isna(ctd['lat']) or pd.isna(ctd['time']): return None, None
    dists = []
    for _, row in df_mvp.iterrows():
        try:
            d = geodesic((ctd['lat'], ctd['lon']), (row['lat'], row['lon'])).km
            dists.append(d)
        except: dists.append(9999)
    min_dist = np.min(dists)
    best_idx = np.argmin(dists)
    best_mvp = df_mvp.iloc[best_idx].copy()
    best_mvp['dist_to_ctd'] = min_dist 
    
    if min_dist > MAX_DIST_KM: return None, None
    if pd.isna(best_mvp['time']): return None, None
    dt_min = abs((best_mvp['time'] - ctd['time']).total_seconds() / 60.0)
    if dt_min > MAX_TIME_MIN: return None, None
    
    indices = [best_idx - 1, best_idx, best_idx + 1]
    indices = [i for i in indices if 0 <= i < len(df_mvp)]
    sequence_df = df_mvp.iloc[indices].copy()
    sequence_df['dist_to_ctd'] = [dists[i] for i in indices]
    return sequence_df, best_mvp

# ==========================================
# 4. FINAL FIGURE
# ==========================================
def plot_figure_1_final_equal(ctd_data, mvp_seq, best_match, save_path):
    
    mvp_data = []
    for _, row in mvp_seq.iterrows():
        try:
            ds = xr.open_dataset(row['path'])
            mvp_data.append({'ds': ds, 'dist': row['dist_to_ctd'], 'name': row['file']})
        except: pass
    if not mvp_data: return

    colors = []
    for m in mvp_data:
        if m['name'] == best_match['file']: colors.append('#d62728') 
        else: colors.append('#7f7f7f') 

    # --- FIGURE ---
    fig = plt.figure(figsize=(12, 12)) 
    
    gs = gridspec.GridSpec(2, 6, height_ratios=[1, 1], hspace=0.25, wspace=0.6)
    
    ax_t = fig.add_subplot(gs[0, 0:2])
    ax_c = fig.add_subplot(gs[0, 2:4])
    ax_s = fig.add_subplot(gs[0, 4:6])
    
    ax_map = fig.add_subplot(gs[1, 0:3]) 

    ax_diff = fig.add_subplot(gs[1, 3:6]) 

    # --- CTD ---
    kw_ctd = {'s': 8, 'color': 'k', 'marker': '.', 'label': 'CTD', 'zorder': 10, 'alpha': 0.9}
    ax_t.scatter(ctd_data['t'], ctd_data['p'], **kw_ctd)
    ax_s.scatter(ctd_data['s'], ctd_data['p'], **kw_ctd)
    if 'c' in ctd_data: ax_c.scatter(ctd_data['c'], ctd_data['p'], **kw_ctd)

    # --- MVP ---
    lats, lons = [], []
    for i, m in enumerate(mvp_data):
        ds = m['ds']
        p = ds['pressure'].values
        t = ds['t1'].values
        c = ds['c1'].values 
        s = ds['s_raw'].values if 's_raw' in ds else (ds['s1'].values if 's1' in ds else np.nan)
        
        is_best = (colors[i] == '#d62728')
        sz = 15 if is_best else 6
        al = 0.6 if is_best else 0.4
        
        ax_t.scatter(t, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        ax_c.scatter(c, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        ax_s.scatter(s, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        
        lats.append(np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values))
        lons.append(np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values))

    for ax in [ax_t, ax_c, ax_s]:
        ax.invert_yaxis()
        ax.set_ylim(Z_MAX_PROFILE, 0)
        ax.grid(True, alpha=0.3)
        
        if ax != ax_t:
            ax.set_yticklabels([])
            ax.set_ylabel("")
            ax.tick_params(axis='y', which='both', left=False)
            
    ax_c.set_yticklabels([])
    ax_c.set_ylabel("")

    ax_t.set_title("(a) Temperature", loc='left', fontweight='bold', fontsize=10)
    ax_t.set_xlabel("°C")
    
    ax_c.set_title("(b) Conductivity", loc='left', fontweight='bold', fontsize=10)
    ax_c.set_xlabel("mS/cm") 
    
    ax_s.set_title("(c) Salinity", loc='left', fontweight='bold', fontsize=10)
    ax_s.set_xlabel("PSU")
    ax_s.set_xlim(SAL_LIMITS[0], SAL_LIMITS[1])

    ax_diff.axvline(0, color='k', linestyle='-', alpha=0.3, lw=1)
    for i, m in enumerate(mvp_data):
        ds = m['ds']
        p_mvp = ds['pressure'].values
        s_mvp = ds['s_raw'].values if 's_raw' in ds else np.nan
        if np.all(np.diff(ctd_data['p']) > 0):
            s_ctd_interp = np.interp(p_mvp, ctd_data['p'], ctd_data['s'])
            diff = s_mvp - s_ctd_interp
            sz = 15 if colors[i] == '#d62728' else 6
            ax_diff.scatter(diff, p_mvp, s=sz, color=colors[i], alpha=0.5, marker='.')

    ax_diff.set_ylim(100, 10) 
    ax_diff.set_title("(e) $\Delta$ Salinity", loc='left', fontweight='bold', fontsize=10)
    ax_diff.set_xlabel("PSU")
    ax_diff.set_xlim(-0.2, 0.2)
    ax_diff.grid(True, alpha=0.3)
    ax_diff.set_ylabel("Pressure (dbar)") 

    # --- MAP ---
    ax_map.plot(ctd_data['lon'], ctd_data['lat'], 'k*', markersize=14, label='CTD')
    for i in range(len(lons)):
        ax_map.plot(lons[i], lats[i], 'o', color=colors[i], markersize=10)
        txt = "0" if colors[i] == '#d62728' else ("-1" if i==0 else "+1")
        ax_map.text(lons[i], lats[i], txt, color='white', ha='center', va='center', fontsize=6, fontweight='bold')

    center_lon, center_lat = ctd_data['lon'], ctd_data['lat']
    FIXED_MARGIN = 0.03 
    ax_map.set_xlim(center_lon - FIXED_MARGIN, center_lon + FIXED_MARGIN)
    ax_map.set_ylim(center_lat - FIXED_MARGIN, center_lat + FIXED_MARGIN)
    
    ax_map.set_title("(d) Spatial Context", loc='left', fontweight='bold', fontsize=10)
    ax_map.set_xlabel("Longitude"); ax_map.set_ylabel("Latitude")
    ax_map.grid(True, ls=':')
    ax_map.set_aspect('equal', 'box')
    
    dt = abs((best_match['time'] - ctd_data['time']).total_seconds()/60)
    info_str = f"D: {best_match['dist_to_ctd']:.2f} km\nT: {dt:.0f} min"
    ax_map.text(0.02, 0.05, info_str, transform=ax_map.transAxes, 
                fontsize=9, bbox=dict(facecolor='white', alpha=0.9, boxstyle='round'))

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='.', color='w', markerfacecolor='k', label='CTD', markersize=12),
        Line2D([0], [0], marker='.', color='w', markerfacecolor='#d62728', label='MVP Match', markersize=12),
        Line2D([0], [0], marker='.', color='w', markerfacecolor='#7f7f7f', label='MVP ±1', markersize=10)
    ]
    ax_s.legend(handles=legend_elements, loc='lower left', fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f" Generating Final Figures...")
    
    df_mvp = load_mvp_catalog_sorted(DIR_MVP_RAW)
    if df_mvp.empty: exit()
    
    ctd_files = sorted(list(DIR_CTD.glob("d*.cnv")))
    
    hits = 0
    for ctd_f in ctd_files:
        ctd = read_ctd_cnv(ctd_f)
        if not ctd: continue
        
        mvp_seq, best_match = find_closest_sequence_strict(ctd, df_mvp)
        
        if mvp_seq is not None:
            print(f"  ✅ {ctd_f.name}")
            savename = OUTDIR / f"Fig1_Equal_{ctd_f.stem}.png"
            try:
                plot_figure_1_final_equal(ctd, mvp_seq, best_match, savename)
                hits += 1
            except Exception as e:
                print(f"     ❌ Error: {e}")

    print(f"\n Done. {hits} figures generated in {OUTDIR}.")

🚀 Generando Figuras Finales Equal Rows...
  ✅ dstation_02.cnv
  ✅ dstation_04.cnv
  ✅ dstation_09.cnv
  ✅ dstation_10.cnv
  ✅ dstation_11.cnv

🏁 Hecho. 5 figuras en figure1_paper_fastswot.


LEG 2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import re
import io
import warnings
import gsw

warnings.filterwarnings("ignore")

# ==========================================
# 1. PARAMETERS
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")
DIR_MVP_RAW = BASE_ROOT / "Data" / "Leg2" / "processed_step1_highres_qc" 
DIR_CTD = Path(r"C:/Users/ASUS/OneDrive - Universitat de les Illes Balears/SWOT/CTD/CTD_data/leg2/HM/")

MAX_DIST_KM = 0.5
MAX_TIME_MIN = 60.0

# Visual Configuration
Z_MAX_PROFILE = 200.0
Z_RANGE_DIFF = (10, 100)
SAL_LIMITS = (38.0, 38.5)

OUTDIR = BASE_ROOT / "Figures" / "figure1_paper_fastswot"
OUTDIR.mkdir(exist_ok=True)

# ==========================================
# 2. READERS
# ==========================================
def parse_cnv_time(line):
    try:
        parts = line.split('=')[1].strip().split('[')[0].strip()
        return pd.to_datetime(parts)
    except: return pd.NaT

def read_ctd_cnv(path):
    try:
        with open(path, 'r', encoding='latin-1') as f: lines = f.readlines()
        lat, lon, time_val = np.nan, np.nan, pd.NaT
        start_idx = 0
        col_map = {}
        name_re = re.compile(r"#\s*name\s*(\d+)\s*=\s*([^:]+)", re.I)
        
        for i, line in enumerate(lines):
            if 'NMEA Latitude' in line:
                parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(parts)>=2: lat = float(parts[0]) + float(parts[1])/60 * (-1 if 'S' in line else 1)
            if 'NMEA Longitude' in line:
                parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(parts)>=2: lon = float(parts[0]) + float(parts[1])/60 * (-1 if 'W' in line else 1)
            if 'start_time' in line: time_val = parse_cnv_time(line)
            m = name_re.search(line)
            if m:
                idx = int(m.group(1)); name = m.group(2).strip().lower()
                if any(x in name for x in ['prdm', 'pres']): col_map['p'] = idx
                elif any(x in name for x in ['t090c', 'temp']): col_map['t'] = idx
                elif any(x in name for x in ['sal00', 'psal']): col_map['s'] = idx
            if '*END*' in line: start_idx = i+1; break
            
        data_str = "".join(lines[start_idx:])
        df = pd.read_csv(io.StringIO(data_str), delim_whitespace=True, header=None, on_bad_lines='skip')
        out = {'lat': lat, 'lon': lon, 'time': time_val, 'file': path.name}
        for k, idx in col_map.items(): 
            out[k] = df.iloc[:, idx].values if idx < df.shape[1] else np.nan
            
        # Recompute Conductivity (mS/cm)
        if 's' in out and 't' in out and 'p' in out:
            out['c'] = gsw.C_from_SP(out['s'], out['t'], out['p'])
            
        return out
    except: return None

def load_mvp_catalog_sorted(mvp_dir):
    files = sorted(list(mvp_dir.glob("*.nc")))
    data = []
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                lat = np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values)
                lon = np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values)
                t_val = pd.NaT
                if 'start_time' in ds: t_val = pd.to_datetime(ds.start_time)
                elif 'time' in ds: 
                    ts = ds.time.values
                    if ts.size > 0: t_val = pd.to_datetime(ts.ravel()[0])
                data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val, 'path': f})
        except: pass
    return pd.DataFrame(data)

# ==========================================
# 3. MATCHING
# ==========================================
def find_closest_sequence_strict(ctd, df_mvp):
    if df_mvp.empty or pd.isna(ctd['lat']) or pd.isna(ctd['time']): return None, None
    dists = []
    for _, row in df_mvp.iterrows():
        try:
            d = geodesic((ctd['lat'], ctd['lon']), (row['lat'], row['lon'])).km
            dists.append(d)
        except: dists.append(9999)
    min_dist = np.min(dists)
    best_idx = np.argmin(dists)
    best_mvp = df_mvp.iloc[best_idx].copy()
    best_mvp['dist_to_ctd'] = min_dist 
    
    if min_dist > MAX_DIST_KM: return None, None
    if pd.isna(best_mvp['time']): return None, None
    dt_min = abs((best_mvp['time'] - ctd['time']).total_seconds() / 60.0)
    if dt_min > MAX_TIME_MIN: return None, None
    
    indices = [best_idx - 1, best_idx, best_idx + 1]
    indices = [i for i in indices if 0 <= i < len(df_mvp)]
    sequence_df = df_mvp.iloc[indices].copy()
    sequence_df['dist_to_ctd'] = [dists[i] for i in indices]
    return sequence_df, best_mvp

# ==========================================
# 4. FINAL FIGURE
# ==========================================
def plot_figure_1_final_equal(ctd_data, mvp_seq, best_match, save_path):
    
    mvp_data = []
    for _, row in mvp_seq.iterrows():
        try:
            ds = xr.open_dataset(row['path'])
            mvp_data.append({'ds': ds, 'dist': row['dist_to_ctd'], 'name': row['file']})
        except: pass
    if not mvp_data: return

    colors = []
    for m in mvp_data:
        if m['name'] == best_match['file']: colors.append('#d62728') 
        else: colors.append('#7f7f7f') 

    fig = plt.figure(figsize=(12, 12)) 
    

    gs = gridspec.GridSpec(2, 6, height_ratios=[1, 1], hspace=0.25, wspace=0.6)
    
    ax_t = fig.add_subplot(gs[0, 0:2])
    ax_c = fig.add_subplot(gs[0, 2:4])
    ax_s = fig.add_subplot(gs[0, 4:6])
    
    ax_map = fig.add_subplot(gs[1, 0:3]) 

    ax_diff = fig.add_subplot(gs[1, 3:6]) 

    # --- CTD ---
    kw_ctd = {'s': 8, 'color': 'k', 'marker': '.', 'label': 'CTD', 'zorder': 10, 'alpha': 0.9}
    ax_t.scatter(ctd_data['t'], ctd_data['p'], **kw_ctd)
    ax_s.scatter(ctd_data['s'], ctd_data['p'], **kw_ctd)
    if 'c' in ctd_data: ax_c.scatter(ctd_data['c'], ctd_data['p'], **kw_ctd)

    # --- MVP ---
    lats, lons = [], []
    for i, m in enumerate(mvp_data):
        ds = m['ds']
        p = ds['pressure'].values
        t = ds['t1'].values
        c = ds['c1'].values 
        s = ds['s_raw'].values if 's_raw' in ds else (ds['s1'].values if 's1' in ds else np.nan)
        
        is_best = (colors[i] == '#d62728')
        sz = 15 if is_best else 6
        al = 0.6 if is_best else 0.4
        
        ax_t.scatter(t, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        ax_c.scatter(c, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        ax_s.scatter(s, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        
        lats.append(np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values))
        lons.append(np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values))

    for ax in [ax_t, ax_c, ax_s]:
        ax.invert_yaxis()
        ax.set_ylim(Z_MAX_PROFILE, 0)
        ax.grid(True, alpha=0.3)
        
        if ax != ax_t:
            ax.set_yticklabels([])
            ax.set_ylabel("")
            ax.tick_params(axis='y', which='both', left=False)
            
    ax_c.set_yticklabels([])
    ax_c.set_ylabel("")

    ax_t.set_title("(a) Temperature", loc='left', fontweight='bold', fontsize=10)
    ax_t.set_xlabel("°C")
    
    ax_c.set_title("(b) Conductivity", loc='left', fontweight='bold', fontsize=10)
    ax_c.set_xlabel("mS/cm") 
    
    ax_s.set_title("(c) Salinity", loc='left', fontweight='bold', fontsize=10)
    ax_s.set_xlabel("PSU")
    ax_s.set_xlim(SAL_LIMITS[0], SAL_LIMITS[1])

    ax_diff.axvline(0, color='k', linestyle='-', alpha=0.3, lw=1)
    for i, m in enumerate(mvp_data):
        ds = m['ds']
        p_mvp = ds['pressure'].values
        s_mvp = ds['s_raw'].values if 's_raw' in ds else np.nan
        if np.all(np.diff(ctd_data['p']) > 0):
            s_ctd_interp = np.interp(p_mvp, ctd_data['p'], ctd_data['s'])
            diff = s_mvp - s_ctd_interp
            sz = 15 if colors[i] == '#d62728' else 6
            ax_diff.scatter(diff, p_mvp, s=sz, color=colors[i], alpha=0.5, marker='.')

    ax_diff.set_ylim(100, 10) 
    ax_diff.set_title("(e) $\Delta$ Salinity", loc='left', fontweight='bold', fontsize=10)
    ax_diff.set_xlabel("PSU")
    ax_diff.set_xlim(-0.2, 0.2)
    ax_diff.grid(True, alpha=0.3)
    ax_diff.set_ylabel("Pressure (dbar)") 

    # --- MAP ---
    ax_map.plot(ctd_data['lon'], ctd_data['lat'], 'k*', markersize=14, label='CTD')
    for i in range(len(lons)):
        ax_map.plot(lons[i], lats[i], 'o', color=colors[i], markersize=10)
        txt = "0" if colors[i] == '#d62728' else ("-1" if i==0 else "+1")
        ax_map.text(lons[i], lats[i], txt, color='white', ha='center', va='center', fontsize=6, fontweight='bold')

    center_lon, center_lat = ctd_data['lon'], ctd_data['lat']
    FIXED_MARGIN = 0.03 
    ax_map.set_xlim(center_lon - FIXED_MARGIN, center_lon + FIXED_MARGIN)
    ax_map.set_ylim(center_lat - FIXED_MARGIN, center_lat + FIXED_MARGIN)
    
    ax_map.set_title("(d) Spatial Context", loc='left', fontweight='bold', fontsize=10)
    ax_map.set_xlabel("Longitude"); ax_map.set_ylabel("Latitude")
    ax_map.grid(True, ls=':')
    ax_map.set_aspect('equal', 'box')
    
    dt = abs((best_match['time'] - ctd_data['time']).total_seconds()/60)
    info_str = f"D: {best_match['dist_to_ctd']:.2f} km\nT: {dt:.0f} min"
    ax_map.text(0.02, 0.05, info_str, transform=ax_map.transAxes, 
                fontsize=9, bbox=dict(facecolor='white', alpha=0.9, boxstyle='round'))

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='.', color='w', markerfacecolor='k', label='CTD', markersize=12),
        Line2D([0], [0], marker='.', color='w', markerfacecolor='#d62728', label='MVP Match', markersize=12),
        Line2D([0], [0], marker='.', color='w', markerfacecolor='#7f7f7f', label='MVP ±1', markersize=10)
    ]
    ax_s.legend(handles=legend_elements, loc='lower left', fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f" Generating Final Figures...")
    
    df_mvp = load_mvp_catalog_sorted(DIR_MVP_RAW)
    if df_mvp.empty: exit()
    
    ctd_files = sorted(list(DIR_CTD.glob("d*.cnv")))
    
    hits = 0
    for ctd_f in ctd_files:
        ctd = read_ctd_cnv(ctd_f)
        if not ctd: continue
        
        mvp_seq, best_match = find_closest_sequence_strict(ctd, df_mvp)
        
        if mvp_seq is not None:
            print(f"  ✅ {ctd_f.name}")
            savename = OUTDIR / f"Fig1_Equal_{ctd_f.stem}.png"
            try:
                plot_figure_1_final_equal(ctd, mvp_seq, best_match, savename)
                hits += 1
            except Exception as e:
                print(f"     ❌ Error: {e}")

    print(f"\n Done. {hits} figures generated in {OUTDIR}.")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


🚀 Generando Figuras Finales Equal Rows...
  ✅ ds2-03.cnv
  ✅ ds4-04.cnv
  ✅ ds4-06.cnv
  ✅ ds5-09.cnv

🏁 Hecho. 4 figuras en figure1_paper_fastswot.


## BIO-SWOT:


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import re
import io
import warnings
import gsw

warnings.filterwarnings("ignore")

# ==========================================
# 1. PARÁMETROS
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing")
DIR_MVP_RAW = BASE_ROOT / "Data" / "processed_step1_highres_qc"
DIR_CTD = Path(r"C:\Users\ASUS\Desktop\BioSWOT_data\CTD\cnv")

# Matching Criteria
MAX_DIST_KM = 3
MAX_TIME_MIN = 120.0

# Visual Configuration
Z_MAX_PROFILE = 350.0       
Z_RANGE_DIFF = (10, 200)     # vertical range for difference plot
SAL_LIMITS = (37.6, 38.7)    # Zoom fixed for Salinity

OUTDIR = BASE_ROOT / "Figures" / "figure1_paper_bioswot"
OUTDIR.mkdir(exist_ok=True)

# ==========================================
# 2. FUNCTIONS
# ==========================================
def parse_cnv_time(line):
    try:
        parts = line.split('=')[1].strip().split('[')[0].strip()
        return pd.to_datetime(parts)
    except: return pd.NaT

def read_ctd_cnv(path):
    try:
        with open(path, 'r', encoding='latin-1') as f: lines = f.readlines()
        lat, lon, time_val = np.nan, np.nan, pd.NaT
        start_idx = 0
        col_map = {}
        name_re = re.compile(r"#\s*name\s*(\d+)\s*=\s*([^:]+)", re.I)
        
        for i, line in enumerate(lines):
            if 'NMEA Latitude' in line:
                parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(parts)>=2: lat = float(parts[0]) + float(parts[1])/60 * (-1 if 'S' in line else 1)
            if 'NMEA Longitude' in line:
                parts = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(parts)>=2: lon = float(parts[0]) + float(parts[1])/60 * (-1 if 'W' in line else 1)
            if 'start_time' in line: time_val = parse_cnv_time(line)
            m = name_re.search(line)
            if m:
                idx = int(m.group(1)); name = m.group(2).strip().lower()
                if any(x in name for x in ['prdm', 'pres']): col_map['p'] = idx
                elif any(x in name for x in ['t090c', 'temp']): col_map['t'] = idx
                elif any(x in name for x in ['sal00', 'psal']): col_map['s'] = idx
                elif any(x in name for x in ['c0', 'c1', 'cond']): col_map['c'] = idx
            if '*END*' in line: start_idx = i+1; break
            
        data_str = "".join(lines[start_idx:])
        df = pd.read_csv(io.StringIO(data_str), delim_whitespace=True, header=None, on_bad_lines='skip')
        out = {'lat': lat, 'lon': lon, 'time': time_val, 'file': path.name}
        for k, idx in col_map.items(): 
            out[k] = df.iloc[:, idx].values if idx < df.shape[1] else np.nan
            
        if 's' in out and 't' in out and 'p' in out:
            try:
                out['c'] = gsw.C_from_SP(out['s'], out['t'], out['p'])
            except: pass
            
        return out
    except: return None

def load_mvp_catalog_sorted(mvp_dir):
    files = sorted(list(mvp_dir.glob("*.nc")))
    data = []
    print(f" Indexing MVP in: {mvp_dir}")
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                lat = np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values)
                lon = np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values)
                t_val = pd.NaT
                if 'start_time' in ds: t_val = pd.to_datetime(ds.start_time)
                elif 'time' in ds: 
                    ts = ds.time.values
                    if ts.size > 0: t_val = pd.to_datetime(ts.ravel()[0])
                data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val, 'path': f})
        except: pass
    return pd.DataFrame(data)

def find_closest_sequence_strict(ctd, df_mvp):
    if df_mvp.empty or pd.isna(ctd['lat']) or pd.isna(ctd['time']): return None, None
    dists = []
    for _, row in df_mvp.iterrows():
        try:
            d = geodesic((ctd['lat'], ctd['lon']), (row['lat'], row['lon'])).km
            dists.append(d)
        except: dists.append(9999)
    min_dist = np.min(dists)
    best_idx = np.argmin(dists)
    best_mvp = df_mvp.iloc[best_idx].copy()
    best_mvp['dist_to_ctd'] = min_dist 
    
    if min_dist > MAX_DIST_KM: return None, None
    if pd.isna(best_mvp['time']): return None, None
    dt_min = abs((best_mvp['time'] - ctd['time']).total_seconds() / 60.0)
    if dt_min > MAX_TIME_MIN: return None, None
    
    indices = [best_idx - 1, best_idx, best_idx + 1]
    indices = [i for i in indices if 0 <= i < len(df_mvp)]
    sequence_df = df_mvp.iloc[indices].copy()
    sequence_df['dist_to_ctd'] = [dists[i] for i in indices]
    return sequence_df, best_mvp

# ==========================================
# 3. STANDARD PLOTTING FUNCTION
# ==========================================
def plot_figure_bioswot_standard(ctd_data, mvp_seq, best_match, save_path):
    
    mvp_data = []
    for _, row in mvp_seq.iterrows():
        try:
            ds = xr.open_dataset(row['path'])
            mvp_data.append({'ds': ds, 'dist': row['dist_to_ctd'], 'name': row['file']})
        except: pass
    if not mvp_data: return

    colors = []
    for m in mvp_data:
        if m['name'] == best_match['file']: colors.append('#d62728') 
        else: colors.append('#7f7f7f') 

    fig = plt.figure(figsize=(12, 12))
    
    gs = gridspec.GridSpec(2, 6, height_ratios=[1, 1], hspace=0.25, wspace=0.6)
    
    ax_t = fig.add_subplot(gs[0, 0:2])
    ax_c = fig.add_subplot(gs[0, 2:4])
    ax_s = fig.add_subplot(gs[0, 4:6])
    
    ax_map = fig.add_subplot(gs[1, 0:3]) 
    ax_diff = fig.add_subplot(gs[1, 3:6]) 

    kw_ctd = {'s': 8, 'color': 'k', 'marker': '.', 'label': 'CTD', 'zorder': 10, 'alpha': 0.9}
    ax_t.scatter(ctd_data['t'], ctd_data['p'], **kw_ctd)
    ax_s.scatter(ctd_data['s'], ctd_data['p'], **kw_ctd)
    
    if 'c' in ctd_data:
        c_val = ctd_data['c']
        if np.nanmean(c_val) < 10: c_val = c_val * 10.0
        ax_c.scatter(c_val, ctd_data['p'], **kw_ctd)

    lats, lons = [], []
    for i, m in enumerate(mvp_data):
        ds = m['ds']
        p = ds['pressure'].values
        t = ds['t1'].values
        c = ds['c1'].values
        if np.nanmean(c) < 10: c = c * 10.0
        
        s = ds['s_raw'].values if 's_raw' in ds else (ds['s1'].values if 's1' in ds else np.nan)
        
        is_best = (colors[i] == '#d62728')
        sz = 15 if is_best else 6
        al = 0.6 if is_best else 0.4
        
        ax_t.scatter(t, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        ax_c.scatter(c, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        ax_s.scatter(s, p, s=sz, color=colors[i], alpha=al, marker='.', edgecolors='none')
        
        lats.append(np.nanmean(ds['lat'].values if 'lat' in ds else ds.latitude.values))
        lons.append(np.nanmean(ds['lon'].values if 'lon' in ds else ds.longitude.values))

    for ax in [ax_t, ax_c, ax_s]:
        ax.invert_yaxis()
        ax.set_ylim(Z_MAX_PROFILE, 0)
        ax.grid(True, alpha=0.3)
        
        if ax != ax_t:
            ax.set_yticklabels([])
            ax.set_ylabel("")
            ax.tick_params(axis='y', which='both', left=False)

    ax_t.set_title("(a) Temperature", loc='left', fontweight='bold', fontsize=10); ax_t.set_xlabel("°C")
    ax_c.set_title("(b) Conductivity", loc='left', fontweight='bold', fontsize=10); ax_c.set_xlabel("mS/cm") 
    ax_s.set_title("(c) Salinity", loc='left', fontweight='bold', fontsize=10); ax_s.set_xlabel("PSU")
    ax_s.set_xlim(SAL_LIMITS[0], SAL_LIMITS[1])

    ax_diff.axvline(0, color='k', linestyle='-', alpha=0.3, lw=1)
    for i, m in enumerate(mvp_data):
        ds = m['ds']
        p_mvp = ds['pressure'].values
        s_mvp = ds['s_raw'].values if 's_raw' in ds else np.nan
        if np.all(np.diff(ctd_data['p']) > 0):
            s_ctd_interp = np.interp(p_mvp, ctd_data['p'], ctd_data['s'])
            diff = s_mvp - s_ctd_interp
            sz = 15 if colors[i] == '#d62728' else 6
            ax_diff.scatter(diff, p_mvp, s=sz, color=colors[i], alpha=0.5, marker='.')

    ax_diff.set_ylim(Z_RANGE_DIFF[1], Z_RANGE_DIFF[0])
    ax_diff.set_title("(e) $\Delta$ Salinity", loc='left', fontweight='bold', fontsize=10)
    ax_diff.set_xlabel("PSU"); ax_diff.set_xlim(-0.2, 0.2); ax_diff.grid(True, alpha=0.3)
    ax_diff.set_ylabel("Pressure (dbar)")

    # --- MAP ---
    ax_map.plot(ctd_data['lon'], ctd_data['lat'], 'k*', markersize=14, label='CTD')
    for i in range(len(lons)):
        ax_map.plot(lons[i], lats[i], 'o', color=colors[i], markersize=10)
        txt = "0" if colors[i] == '#d62728' else ("-1" if i==0 else "+1")
        ax_map.text(lons[i], lats[i], txt, color='white', ha='center', va='center', fontsize=6, fontweight='bold')

    center_lon, center_lat = ctd_data['lon'], ctd_data['lat']
    
    MARGIN_LAT = 0.025 
    margin_lon = MARGIN_LAT / np.cos(np.deg2rad(center_lat))
    
    ax_map.set_xlim(center_lon - margin_lon, center_lon + margin_lon)
    ax_map.set_ylim(center_lat - MARGIN_LAT, center_lat + MARGIN_LAT)
    
    ax_map.set_title("(d) Spatial Context", loc='left', fontweight='bold', fontsize=10)
    ax_map.set_xlabel("Longitude"); ax_map.set_ylabel("Latitude")
    ax_map.grid(True, ls=':')
    
    dt = abs((best_match['time'] - ctd_data['time']).total_seconds()/60)
    info_str = f"D: {best_match['dist_to_ctd']:.2f} km\nT: {dt:.0f} min"
    ax_map.text(0.02, 0.05, info_str, transform=ax_map.transAxes, 
                fontsize=9, bbox=dict(facecolor='white', alpha=0.9, boxstyle='round'))

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='.', color='w', markerfacecolor='k', label='CTD', markersize=12),
        Line2D([0], [0], marker='.', color='w', markerfacecolor='#d62728', label='MVP Match', markersize=12),
        Line2D([0], [0], marker='.', color='w', markerfacecolor='#7f7f7f', label='MVP ±1', markersize=10)
    ]
    ax_s.legend(handles=legend_elements, loc='lower left', fontsize=8)


    plt.savefig(save_path, dpi=300) 
    plt.close()

# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f" Generating Figures BioSWOT...")
    
    df_mvp = load_mvp_catalog_sorted(DIR_MVP_RAW)
    if df_mvp.empty: exit()
    
    ctd_files = sorted(list(DIR_CTD.glob("d*.cnv"))) 
    if not ctd_files: ctd_files = sorted(list(DIR_CTD.glob("*.cnv")))
    
    hits = 0
    for ctd_f in ctd_files:
        ctd = read_ctd_cnv(ctd_f)
        if not ctd: continue
        
        mvp_seq, best_match = find_closest_sequence_strict(ctd, df_mvp)
        
        if mvp_seq is not None:
            print(f"  ✅ {ctd_f.name}")
            savename = OUTDIR / f"BioSWOT_Std_{ctd_f.stem}.png"
            try:
                plot_figure_bioswot_standard(ctd, mvp_seq, best_match, savename)
                hits += 1
            except Exception as e:
                print(f"     ❌ Error: {e}")

    print(f"\n Done. {hits} figures in {OUTDIR}.")

🚀 Generando Figuras BioSWOT (Tamaño Fijo + Cond OK)...
📂 Indexando MVP en: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\processed_step1_highres_qc
  ✅ bioswot2023_0017.cnv
  ✅ bioswot2023_0018.cnv
  ✅ bioswot2023_0019.cnv
  ✅ bioswot2023_0038.cnv

🏁 Hecho. 4 figuras en C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Figures\figure1_paper_bioswot.
